## Задача 1. My heart will go on

Датасет **titanic** из библиотеки `Seaborn` содержит информацию о пассажирах легендарного корабля Титаник, который затонул в 1912 году после столкновения с айсбергом. Этот набор данных часто используется для обучения и тестирования алгоритмов машинного обучения, особенно в задачах бинарной классификации (выжил / не выжил).

**Описание данных**

| Поле         | Тип      | Описание |
|--------------|----------|----------|
| `survived`   | int      | Выжил (1) или не выжил (0) |
| `pclass`     | int      | Класс билета (1, 2, 3) |
| `sex`        | str      | Пол (`male`/`female`) |
| `age`        | float    | Возраст |
| `sibsp`      | int      | Количество братьев/сестёр/супругов на борту |
| `parch`      | int      | Количество родителей/детей на борту |
| `fare`       | float    | Стоимость билета |
| `embarked`   | str      | Порт посадки (`C`=Cherbourg, `Q`=Queenstown, `S`=Southampton) |
| `class`      | str      | Класс билета (`First`, `Second`, `Third`) |
| `who`        | str      | Категория: `man`, `woman` или `child` |
| `adult_male` | bool     | Является ли взрослым мужчиной |
| `deck`       | str      | Палуба |
| `embark_town`| str      | Название порта посадки |
| `alive`      | str      | Выжил (`yes`/`no`) |
| `alone`      | bool     | Путешествовал один |

**Загрузка датасета**

In [1]:
import seaborn as sns
import numpy as np
import pandas as pd

titanic_data = sns.load_dataset("titanic")
titanic_data.sample(10)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
85,1,3,female,33.0,3,0,15.8500,S,Third,woman,False,NaN,Southampton,yes,False
622,1,3,male,20.0,1,1,15.7417,C,Third,man,True,NaN,Cherbourg,yes,False
743,0,3,male,24.0,1,0,16.1000,S,Third,man,True,NaN,Southampton,no,False
709,1,3,male,NaN,1,1,15.2458,C,Third,man,True,NaN,Cherbourg,yes,False
258,1,1,female,35.0,0,0,512.3292,C,First,woman,False,NaN,Cherbourg,yes,True
161,1,2,female,40.0,0,0,15.7500,S,Second,woman,False,NaN,Southampton,yes,True
74,1,3,male,32.0,0,0,56.4958,S,Third,man,True,NaN,Southampton,yes,True
884,0,3,male,25.0,0,0,7.0500,S,Third,man,True,NaN,Southampton,no,True
415,0,3,female,NaN,0,0,8.0500,S,Third,woman,False,NaN,Southampton,no,True
29,0,3,male,NaN,0,0,7.8958,S,Third,man,True,NaN,Southampton,no,True


**Задача**

Ниже описаны 10 небольших заданий, которые вам необходимо решить.

**Подсказка**:

В некоторых заданиях вам может быть полезен метод `value_counts`.

### Часть 1

Определите число пропущенных данных для каждого столбца таблицы `titanic_data`.

In [2]:
empty = titanic_data.isnull().values
# empty = empty.astype(int)
empty = empty.sum(axis=0)
number_of_empty = pd.Series(
    index= titanic_data.columns,
    data= empty
)
number_of_empty

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

### Часть 2

Удалите все столбцы, количество пропусков в которых превышает половину количества строк в таблице.

После того, как вы удалите все столбцы, нарушающие описанное условие, удалите все строки, количество пропусков в которых превышает половину количества столбцов.

In [3]:
titanic_data = titanic_data.dropna(thresh= (titanic_data.values.shape[0] + 1) // 2, axis= "columns")
titanic_data = titanic_data.dropna(thresh= (titanic_data.values.shape[1]+1) // 2)

### Часть 3

Если вы сделали все правильно, больше всего пропусков должно остаться в столбце `"age"` - 177. Их необходимо заполнить. Заполним пропуски следующим образом:
- Если значение столбца `"who"="man"`, пропуск необходимо заполнить медианным значением известных возрастов мужчин, округленным до ближайшего целого числа;
- Если значение столбца `"who"="woman"`, пропуск необходимо заполнить медианным значением известных возрастов женщин, округленным до ближайшего целого числа;
- Если значение столбца `"who"="child"`, пропуск необходимо заполнить медианным значением известных возрастов детей, округленным до ближайшего целого числа;

In [4]:
rows_to_fill_men = ((titanic_data["age"].isna()) & (titanic_data["who"] == "man"))
rows_to_fill_women = ((titanic_data["age"].isna()) & (titanic_data["who"] == "woman"))
rows_to_fill_children = ((titanic_data["age"].isna()) & (titanic_data["who"] == "child"))

men_middle_age = round(titanic_data[titanic_data["who"] == "man"]["age"].dropna().median())
women_middle_age = round(titanic_data[titanic_data["who"] == "woman"]["age"].dropna().median())
children_middle_age = round(titanic_data[titanic_data["who"] == "child"]["age"].dropna().median())
titanic_data.loc[rows_to_fill_children, "age"] = children_middle_age
titanic_data.loc[rows_to_fill_men, "age"] = men_middle_age
titanic_data.loc[rows_to_fill_women, "age"] = women_middle_age

### Часть 4

Удалите все строки, в которых осталось больше одного пропуска. Если вы все сделали правильно, после этого действия в таблице не должно остаться пропусков.

In [5]:
titanic_data = titanic_data.dropna(thresh=(titanic_data.values.shape[1] - 1))

### Часть 5

Определите название города, из которого отправилось больше всего пассажиров.

In [6]:
cities = titanic_data["embarked"].dropna().values
unique, counts = np.unique(cities, return_counts=True)
if unique[counts.argmax()] == 'S':
    print("Southampton")
elif unique[counts.argmax()] == 'Q':
    print("Queenstown")
else:
    print("Cherbough")


Southampton


### Часть 6

Определите процент выживших пассажиров от числа пассажиров, оставшихся в таблице после очистки данных. Ответ округлите до 2 знаков после запятой.

In [7]:
survived_ones = titanic_data[titanic_data["survived"] == 1]
round((len(survived_ones.index) / len(titanic_data.index)) * 100, 2)

38.25

### Часть 7

Определите число выживших пассажиров для каждого пункта отправления. В ответе должен получиться объект типа `pd.Series`, индексы которого - названия пунктов отправления, а значения - число выживших пассажиров.

In [8]:
Southampton_city = survived_ones[titanic_data["embarked"] == "S"]
Queenstown_city = survived_ones[titanic_data["embarked"] == "Q"]
Cherbough_city = survived_ones[titanic_data["embarked"] == "C"]

cities_and_survivors = pd.Series(
    index= ["Southampton", "Queenstown", "Cherbough"],
    data= [len(Southampton_city), len(Queenstown_city), len(Cherbough_city)]
)

cities_and_survivors

/var/folders/27/dvqvlln90m15hk5wbzsb7lj80000gn/T/ipykernel_10356/87840810.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  Southampton_city = survived_ones[titanic_data["embarked"] == "S"]
/var/folders/27/dvqvlln90m15hk5wbzsb7lj80000gn/T/ipykernel_10356/87840810.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  Queenstown_city = survived_ones[titanic_data["embarked"] == "Q"]
/var/folders/27/dvqvlln90m15hk5wbzsb7lj80000gn/T/ipykernel_10356/87840810.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  Cherbough_city = survived_ones[titanic_data["embarked"] == "C"]


Southampton    217
Queenstown      30
Cherbough       93
dtype: int64

### Часть 8

Определите процент выживших пассажиров в каждом классе. Значения округлите до 2 знаков после запятой. В ответе должен получиться объект типа `pd.Series`, индексы которого - названия классов, а значения - процент выживших пассажиров.

In [9]:
first_class_survived = survived_ones[survived_ones["class"] == "First"]
second_class_survived = survived_ones[survived_ones["class"] == "Second"]
third_class_survived = survived_ones[survived_ones["class"] == "Third"]

percent_first = round(len(first_class_survived) / len(titanic_data[titanic_data["class"] == "First"]) * 100, 2)
percent_second = round(len(second_class_survived) / len(titanic_data[titanic_data["class"] == "Second"]) * 100, 2)
percent_third = round(len(third_class_survived) / len(titanic_data[titanic_data["class"] == "Third"]) * 100, 2)



statistics = pd.Series(
    index=["First", "Second", "Third"],
    data= [percent_first, percent_second, percent_third]
)
statistics

First     62.62
Second    47.28
Third     24.24
dtype: float64

### Часть 9

Будем считать, что пассажиры, купившие билет **НЕ МЕНЕЕ** чем за $100, считаются богатыми. Определите процент выживших среди богатых пассажиров. Ответ округлите до 2 знаков после запятой. В ответе должно получиться число. 

In [10]:
rich_ones = titanic_data[titanic_data["fare"] >= 100]
rich_ones_number = len(rich_ones.index)
rich_ones_survived = survived_ones[survived_ones["fare"] >= 100]
percentage_of_rich_survivors = round(len(rich_ones_survived.index) / rich_ones_number * 100, 2)
percentage_of_rich_survivors

73.58

### Часть 10

Определите количество детей, путешествовавших в одиночку.

In [11]:
children = titanic_data[titanic_data["who"] == "child"]
children = children[children["alone"]]
len(children.index)

6

Какие выводы вы можете сделать о выживших пассажирах Титаника? 